In [1]:
# notebooks/02_explainability.ipynb
import os
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set(style="whitegrid")

# Paths - adjust if your folder differs
FEATURES_CSV = "../data/airbnb_features.csv"
LGB_MODEL = "../models/model_lightgbm.joblib"
OUTPUT_DIR = "../outputs/explainability"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Files expected:")
print(" Features:", FEATURES_CSV)
print(" LightGBM model:", LGB_MODEL)
print(" Output folder:", OUTPUT_DIR)

C:\Users\DELL\anaconda3\envs\airbnb_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Files expected:
 Features: ../data/airbnb_features.csv
 LightGBM model: ../models/model_lightgbm.joblib
 Output folder: ../outputs/explainability


In [2]:
# Load features and trained model
df = pd.read_csv(FEATURES_CSV, low_memory=False)
print("Features shape:", df.shape)

# Ensure target column exists and separate X
if "price" in df.columns:
    X = df.drop(columns=["price"])
else:
    X = df.copy()

# Load the LightGBM model saved earlier
model = joblib.load(LGB_MODEL)
print("Model loaded:", type(model))

# For SHAP plotting, sample up to 2000 rows (adjust if you want more)
sample_n = 2000
X_sample = X.sample(n=min(sample_n, len(X)), random_state=42).reset_index(drop=True)
print("Using sample for SHAP:", X_sample.shape)

Features shape: (73367, 14)
Model loaded: <class 'lightgbm.basic.Booster'>
Using sample for SHAP: (2000, 13)


In [3]:
# Create an explainer. Works for LightGBM booster and sklearn wrappers.
# shap.TreeExplainer handles LightGBM models (Booster or sklearn API)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)  # returns array shaped (n_samples, n_features)
print("SHAP values shape:", np.array(shap_values).shape)

SHAP values shape: (2000, 13)


In [4]:
# Summary plot (beeswarm) — shows per-feature impact distribution
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP summary (beeswarm) — feature impact on price")
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_beeswarm.png"), bbox_inches="tight")
plt.close()
print("Saved shap_summary_beeswarm.png")

Saved shap_summary_beeswarm.png


In [5]:
# Compute mean absolute SHAP per feature and save as CSV
if isinstance(shap_values, list):  # some SHAP versions return list for multiclass
    shap_arr = np.array(shap_values[0])
else:
    shap_arr = np.array(shap_values)

mean_abs_shap = np.mean(np.abs(shap_arr), axis=0)
feat_importance = pd.DataFrame({
    "feature": X_sample.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

feat_importance.to_csv(os.path.join(OUTPUT_DIR, "shap_feature_importance.csv"), index=False)
print("Saved shap_feature_importance.csv")

# Bar chart of top 20 important features
top_k = 20
plt.figure(figsize=(10,6))
sns.barplot(x="mean_abs_shap", y="feature", data=feat_importance.head(top_k))
plt.title("Top features by mean |SHAP value|")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_feature_importance_bar.png"), bbox_inches="tight")
plt.close()
print("Saved shap_feature_importance_bar.png")
feat_importance.head(20)

Saved shap_feature_importance.csv
Saved shap_feature_importance_bar.png


,feature,mean_abs_shap
0,is_entire_place,25.446161
1,bedrooms,21.665477
2,accommodates,21.229108
3,te_neighbourhood,19.452425
4,neighbourhood_median_price,14.512460
5,bathrooms,11.728041
6,te_property_type,4.494930
7,host_age_days,2.961342
8,amenities_count,2.912901
9,beds,1.967061


In [6]:
# Dependence plot for the top feature (shows interaction)
top_feature = feat_importance.loc[0, "feature"]
print("Top feature:", top_feature)

plt.figure(figsize=(8,5))
shap.dependence_plot(top_feature, shap_values, X_sample, show=False)
plt.title(f"SHAP dependence plot — {top_feature}")
plt.savefig(os.path.join(OUTPUT_DIR, f"shap_dependence_{top_feature}.png"), bbox_inches="tight")
plt.close()
print(f"Saved shap_dependence_{top_feature}.png")

Top feature: is_entire_place
Saved shap_dependence_is_entire_place.png


<Figure size 800x500 with 0 Axes>

In [7]:
# Save SHAP values (sample) with feature columns
shap_df = pd.DataFrame(shap_arr, columns=X_sample.columns)
shap_df.to_csv(os.path.join(OUTPUT_DIR, "shap_values_sample.csv"), index=False)
print("Saved shap_values_sample.csv (sample size {})".format(shap_df.shape[0]))

Saved shap_values_sample.csv (sample size 2000)


In [8]:
# Quick textual summary you can copy into your 1-2 page report
top_feats = feat_importance.head(5)
print("Top 5 features by mean |SHAP| (impact on price):")
print(top_feats.to_string(index=False))

print("\nNotes:")
print("- SHAP summary plot (beeswarm) shows direction and distribution of feature impacts.")
print("- Mean |SHAP| ranks features by overall importance; include top 3 in your 1-2 page report.")
print("- Save the images from ../outputs/explainability/ into your report appendix.")

Top 5 features by mean |SHAP| (impact on price):
                   feature  mean_abs_shap
           is_entire_place      25.446161
                  bedrooms      21.665477
              accommodates      21.229108
          te_neighbourhood      19.452425
neighbourhood_median_price      14.512460

Notes:
- SHAP summary plot (beeswarm) shows direction and distribution of feature impacts.
- Mean |SHAP| ranks features by overall importance; include top 3 in your 1-2 page report.
- Save the images from ../outputs/explainability/ into your report appendix.
